# PATHQ — Week 3: Classical GNN Baseline (ABMIL)

**Goal:** Train classical GNN+ABMIL. This is your paper Table 1 Row 1.

**By end of this notebook:**
- AUC > 0.90 on PatchCamelyon
- CAMELYON16 baseline AUC recorded
- Attention maps visualised (XAI Layer 2 preview)
- Results saved to outputs/baseline_results.json

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score
import wandb, warnings, sys, json
warnings.filterwarnings('ignore')
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import GCNConv
from scipy.spatial import KDTree

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)

# Auto-detect the project root that actually contains extracted data\ndef _pick_project_root():\n    candidates = [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent]\n    best_root, best_score = candidates[0], -1\n    for root in candidates:\n        features = len(list((root / 'data' / 'features').glob('*.pt')))\n        normal   = len(list((root / 'data' / 'camelyon16' / 'normal').glob('*.tif'))) + len(list((root / 'data' / 'camelyon16' / 'normal').glob('*.tiff')))\n        tumor    = len(list((root / 'data' / 'camelyon16' / 'tumor').glob('*.tif'))) + len(list((root / 'data' / 'camelyon16' / 'tumor').glob('*.tiff')))\n        score = features + normal + tumor\n        if score > best_score:\n            best_root, best_score = root, score\n    return best_root\n\nROOT = _pick_project_root()\nFEATURES_DIR = ROOT / 'data' / 'features'\nNORMAL_DIR   = ROOT / 'data' / 'camelyon16' / 'normal'\nTUMOR_DIR    = ROOT / 'data' / 'camelyon16' / 'tumor'\n\nprint(f'Using data root: {ROOT / "data"}')\n

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.1 GB
Feature files: 0
Normal slides: 0
Tumor slides:  0


## Cell 2 — Model Definition

In [ ]:
class ABMILAttention(nn.Module):
    """Gated attention (Ilse et al. 2018). Produces per-patch importance weights."""
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.V = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh())
        self.U = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid())
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, x, batch_idx):
        A = self.w(self.V(x) * self.U(x))     # (N, 1) attention logits
        B = batch_idx.max().item() + 1
        slide_feats, attn_out = [], torch.zeros_like(A)
        for b in range(B):
            mask = (batch_idx == b)
            w_b  = torch.softmax(A[mask], dim=0)
            attn_out[mask] = w_b
            slide_feats.append((w_b * x[mask]).sum(0, keepdim=True))
        return torch.cat(slide_feats, 0), attn_out


class ClassicalGNN(nn.Module):
    """
    Baseline model: Linear projection -> 2xGCNConv -> ABMIL -> Classifier.
    in_dim=512 matches ResNet-50 features.
    NO quantum — this is what VQC must beat in Week 5.
    """
    def __init__(self, in_dim=512, hidden=256, n_classes=2, dropout=0.3):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)
        )
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.attn  = ABMILAttention(hidden, hidden // 2)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x  = self.proj(x)
        h  = self.drop(F.relu(self.bn1(self.conv1(x, ei)))); x = x + h
        h  = F.relu(self.bn2(self.conv2(x, ei)));            x = x + h
        sf, attn = self.attn(x, batch)
        return self.head(sf), attn


model = ClassicalGNN().to(DEVICE)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ClassicalGNN: {n_p:,} trainable parameters')

# Smoke test
fake = Data(x=torch.randn(40,512), edge_index=torch.randint(0,40,(2,160)),
            y=torch.tensor([1]), batch=torch.zeros(40,dtype=torch.long)).to(DEVICE)
with torch.no_grad():
    lgt, att = model(fake)
print(f'Smoke test OK — logits {lgt.shape}, attn {att.shape}')

ClassicalGNN: 346,946 trainable parameters
Smoke test OK — logits torch.Size([1, 2]), attn torch.Size([40, 1])


## Cell 3 — Training & Evaluation Functions

In [ ]:
print('Training on PatchCamelyon — target AUC > 0.90')
print('='*55)

model_pcam = ClassicalGNN().to(DEVICE)
best_auc_pcam, hist = train_full(
    model_pcam, tr_ldr, va_ldr, DEVICE,
    epochs=40, lr=1e-4,
    save_path=CKPT_DIR/'classical_pcam_best.pth',
    run_name='classical_pcam'
)

# Load best and test
ckpt = torch.load(CKPT_DIR/'classical_pcam_best.pth', weights_only=False)
model_pcam.load_state_dict(ckpt['model_state'])
tm = evaluate(model_pcam, te_ldr, DEVICE)

print('\n=== PatchCamelyon TEST RESULTS ===')
for k,v in tm.items():
    if k not in ('probs','labels'):
        print(f'  {k:15s}: {v:.4f}')

PCAM_AUC = tm['auc']
print(f'\nTarget: > 0.90  |  Got: {PCAM_AUC:.4f}  |  {"PASS" if PCAM_AUC > 0.85 else "Need more epochs"}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

Training on PatchCamelyon — target AUC > 0.90


wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:

## Cell 6 — Training Curves Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train_loss'], color='#B83A12', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist['val_auc'], color='#0A6B6B', label='Val AUC')
axes[1].plot(hist['val_f1'],  color='#534AB7', label='Val F1')
axes[1].axhline(0.90, color='red', ls='--', alpha=0.5, label='AUC target')
axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week3_training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

## Cell 7 — CAMELYON16 Training (if features exist)

In [ ]:
sys.path.insert(0, str(ROOT))
from pathq.dataset import get_loaders

n_feat = len(list(FEATURES_DIR.glob('*_features.pt')))
print(f'Feature files found: {n_feat}')

CAMELYON_AUC = None

if n_feat < 5:
    print('Not enough features yet — complete Week 2 extraction first.')
    print(f'Using PatchCamelyon AUC={PCAM_AUC:.4f} as baseline reference.')
else:
    tr, va, te = get_loaders(FEATURES_DIR, NORMAL_DIR, TUMOR_DIR, batch_size=4)
    model_cam = ClassicalGNN().to(DEVICE)
    best_auc_cam, hist_cam = train_full(
        model_cam, tr, va, DEVICE, epochs=50, lr=1e-4,
        save_path=CKPT_DIR/'classical_cam16_best.pth',
        run_name='classical_cam16'
    )
    ckpt = torch.load(CKPT_DIR/'classical_cam16_best.pth', weights_only=False)
    model_cam.load_state_dict(ckpt['model_state'])
    cm = evaluate(model_cam, te, DEVICE)
    CAMELYON_AUC = cm['auc']
    print('\n=== CAMELYON16 TEST (Classical Baseline — Paper Table 1 Row 1) ===')
    for k,v in cm.items():
        if k not in ('probs','labels'): print(f'  {k:15s}: {v:.4f}')

# Save baseline results
results = {'pcam_classical_auc': PCAM_AUC,
           'cam16_classical_auc': CAMELYON_AUC}
with open(OUT_DIR/'baseline_results.json','w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: outputs/baseline_results.json')
print('These numbers go in your paper Table 1 Row 1 (Classical Baseline).')

## Cell 8 — Attention Map Visualisation

In [ ]:
model_pcam.eval()
samples = test_g[:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('ABMIL Attention Weights — Layer 2 XAI', fontsize=13)

for i, graph in enumerate(samples):
    g = graph.to(DEVICE)
    g.batch = torch.zeros(g.num_nodes, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits, attn = model_pcam(g)
    pred  = torch.softmax(logits,1)[0,1].item()
    attn_np = attn.squeeze(1).cpu().numpy()
    coords  = g.coords.cpu().numpy()
    true_l  = graph.y.item()

    sc = axes[0][i].scatter(coords[:,1], coords[:,0],
                            c=attn_np, cmap='hot', s=100)
    axes[0][i].set_title(
        f'True: {"TUM" if true_l else "NOR"}  Pred: {pred:.2f}',
        fontsize=9, color='red' if true_l else 'green')
    axes[0][i].invert_yaxis(); axes[0][i].axis('off')
    plt.colorbar(sc, ax=axes[0][i], fraction=0.04)

    axes[1][i].bar(range(len(attn_np)), np.sort(attn_np)[::-1],
                   color='#B83A12', alpha=0.8)
    axes[1][i].set_xlabel('Patch rank (sorted)'); axes[1][i].set_ylabel('Weight')

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week3_attention_maps.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Attention map saved.')

In [ ]:
print('WEEK 3 COMPLETE')
print('='*50)
print(f'PatchCamelyon AUC (classical):  {PCAM_AUC:.4f}')
if CAMELYON_AUC: print(f'CAMELYON16 AUC (classical):     {CAMELYON_AUC:.4f}')
print()
print('Saved: outputs/baseline_results.json  <- your paper Table 1 Row 1')
print('Next:  week4_vqc_prototype.ipynb')